## K-Nearest Neighbors (KNN)

The flow is:

1. Train KNN on transformed data.
2. Train KNN with different datasets:
   - ADASYN
   - SMOTE
   - SMOTE-ENN
   - SMOTE-TOMEK
3. Compare all experiments using evaluation metrics.
4. Tune KNN hyperparameters.
5. Select the best KNN model.
6. Save the best model.
7. Evaluate the final saved model on test data.

In [60]:
import sys
from pathlib import Path

project_root = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src").exists()),
    None,
)

if project_root is not None and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [61]:
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
import joblib
from src.diabetes_prediction.transformation.transformation import DataTransformation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
)

### Load Data

In [62]:
train_transformed = pd.read_csv('../../data/preprocessed/train_preprocessed.csv')
x_train_transformed = train_transformed.drop("diabetes", axis=1)
y_train_transformed = train_transformed["diabetes"]

val = pd.read_csv('../../data/preprocessed/validation_preprocessed.csv')
x_val = val.drop("diabetes", axis=1)
y_val = val["diabetes"]

print(f'x_train_transformed: {train_transformed.shape}')
train_transformed.head()

x_train_transformed: (67139, 21)


,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,glucose_hba1c_interaction,...,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag,diabetes
0,0.0,0.0,0.0,0.0,0.0,0.612112,-0.637209,0.000000,-0.677966,-0.459103,...,-0.056612,-0.232854,-0.081827,0,0,0,0,0,0,0
1,0.0,0.0,1.0,0.0,0.0,0.737237,0.818605,0.500000,0.000000,0.411609,...,0.658534,1.018393,0.557564,0,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.399399,-0.229457,0.000000,0.254237,0.382586,...,-0.339001,0.014118,-0.070409,1,0,0,0,0,0,0
3,0.0,1.0,0.0,0.0,0.0,0.874875,1.485271,0.500000,0.254237,0.668865,...,1.258590,1.470922,1.050428,0,0,0,0,1,0,0
4,0.0,0.0,0.0,1.0,0.0,0.687187,-0.395349,-1.642857,1.016949,-0.142480,...,0.148131,-1.008759,1.078972,1,0,0,0,0,0,0


In [63]:
train_ADASYN = pd.read_csv('../../data/imbalance_resolve/ADASYN.csv')
x_train_ADASYN = train_ADASYN.drop("diabetes_target", axis=1)
y_train_ADASYN = train_ADASYN["diabetes_target"]
print(f'train_ADASYN: {train_ADASYN.shape}')

train_smote_enn = pd.read_csv('../../data/imbalance_resolve/train_smote_enn.csv')
x_train_smote_enn = train_smote_enn.drop("diabetes_target", axis=1)
y_train_smote_enn = train_smote_enn["diabetes_target"]
print(f'train_smote_enn: {train_smote_enn.shape}')

train_smote_tomek = pd.read_csv('../../data/imbalance_resolve/train_smote_tomek.csv')
x_train_smote_tomek = train_smote_tomek.drop("diabetes_target", axis=1)
y_train_smote_tomek = train_smote_tomek["diabetes_target"]
print(f'train_smote_tomek: {train_smote_tomek.shape}')

train_smote = pd.read_csv('../../data/imbalance_resolve/train_smote.csv')
x_train_smote = train_smote.drop("diabetes_target", axis=1)
y_train_smote = train_smote["diabetes_target"]
print(f'train_smote: {train_smote.shape}')

train_ADASYN: (122666, 16)
train_smote_enn: (112303, 16)
train_smote_tomek: (121658, 16)
train_smote: (112303, 16)


### Evaluation Metrics
- Accuracy
- **Precision**: When I say diabetic, how often am I right?, TP / (TP + FP)
- **Recall**: How many diabetics did I catch?, TP / (TP + FN) :: **most important**
- **F1-score**
- ROC-AUC
- **PR-AUC**
- **MCC**

`it is more important to us to say that a patient has diabetes when he has, and missing that would be bad that is why recall is most important here`

#### Helper functions for model evaluation

In [64]:
def evaluate_model(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]

    y_pred = (y_proba >= threshold).astype(int)

    results = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred),
        "F1-score": f1_score(y, y_pred),
        "ROC-AUC": roc_auc_score(y, y_proba),
        "PR-AUC": average_precision_score(y, y_proba),
        "MCC": matthews_corrcoef(y, y_pred)
    }

    return results

def print_results(title, results):
    print(f"========== {title} ==========")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")
    print()


def evaluate_and_store(results_table, name, model, X, y, threshold=0.5):
    results = evaluate_model(model, X, y, threshold=threshold)
    print_results(name, results)

    row = {"Experiment": name}
    row.update(results)
    results_table.append(row)

## Experiment 1 — KNN on Transformed Train Data

In [65]:
knn_results_table = []

model_knn_transformed = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="minkowski",
    n_jobs=-1,
)

model_knn_transformed.fit(x_train_transformed, y_train_transformed)

evaluate_and_store(
    knn_results_table,
    "KNN - transformed",
    model_knn_transformed,
    x_val,
    y_val,
)

========== KNN - transformed ==========
Accuracy: 0.9595
Precision: 0.8684
Recall: 0.6384
F1-score: 0.7358
ROC-AUC: 0.9150
PR-AUC: 0.7652
MCC: 0.7244



## Experiment 2 — KNN on Transformed Train Data with Distance Weights

In [66]:
model_knn_transformed_w = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="minkowski",
    n_jobs=-1,
)

model_knn_transformed_w.fit(x_train_transformed, y_train_transformed)

evaluate_and_store(
    knn_results_table,
    "KNN - transformed - weights distance",
    model_knn_transformed_w,
    x_val,
    y_val,
)

========== KNN - transformed - weights distance ==========
Accuracy: 0.9577
Precision: 0.8360
Recall: 0.6494
F1-score: 0.7310
ROC-AUC: 0.9149
PR-AUC: 0.7859
MCC: 0.7151



## Experiment 3 — KNN on ADASYN Train Data

In [67]:
model_knn_adasyn = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="minkowski",
    n_jobs=-1,
)

model_knn_adasyn.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    knn_results_table,
    "KNN - ADASYN",
    model_knn_adasyn,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== KNN - ADASYN ==========
Accuracy: 0.9025
Precision: 0.4725
Recall: 0.8844
F1-score: 0.6159
ROC-AUC: 0.9465
PR-AUC: 0.7276
MCC: 0.6025



## Experiment 4 — KNN on SMOTE Train Data

In [68]:
model_knn_smote = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="minkowski",
    n_jobs=-1,
)

model_knn_smote.fit(x_train_smote, y_train_smote)

evaluate_and_store(
    knn_results_table,
    "KNN - SMOTE",
    model_knn_smote,
    x_val[x_train_smote.columns],
    y_val,
)

========== KNN - SMOTE ==========
Accuracy: 0.8964
Precision: 0.4564
Recall: 0.9017
F1-score: 0.6061
ROC-AUC: 0.9305
PR-AUC: 0.5311
MCC: 0.5964



## Experiment 5 — KNN on SMOTE-ENN Train Data

In [69]:
model_knn_smote_enn = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="minkowski",
    n_jobs=-1,
)

model_knn_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

evaluate_and_store(
    knn_results_table,
    "KNN - SMOTE-ENN",
    model_knn_smote_enn,
    x_val[x_train_smote_enn.columns],
    y_val,
)

========== KNN - SMOTE-ENN ==========
Accuracy: 0.8964
Precision: 0.4564
Recall: 0.9017
F1-score: 0.6061
ROC-AUC: 0.9305
PR-AUC: 0.5311
MCC: 0.5964



## Experiment 6 — KNN on SMOTE-TOMEK Train Data

In [70]:
model_knn_smote_tomek = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="minkowski",
    n_jobs=-1,
)

model_knn_smote_tomek.fit(x_train_smote_tomek, y_train_smote_tomek)

evaluate_and_store(
    knn_results_table,
    "KNN - SMOTE-TOMEK",
    model_knn_smote_tomek,
    x_val[x_train_smote_tomek.columns],
    y_val,
)

========== KNN - SMOTE-TOMEK ==========
Accuracy: 0.9217
Precision: 0.5352
Recall: 0.8664
F1-score: 0.6617
ROC-AUC: 0.9455
PR-AUC: 0.7431
MCC: 0.6432



## KNN Experiments Evaluation Results

In [71]:
df = pd.DataFrame(knn_results_table)
df.head(10)

,Experiment,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC,MCC
0,KNN - transformed,0.959477,0.868449,0.638365,0.735841,0.915031,0.765179,0.724385
1,KNN - transformed - weights distance,0.957740,0.836032,0.649371,0.730973,0.914886,0.785911,0.715098
2,KNN - ADASYN,0.902481,0.472491,0.884434,0.615932,0.946535,0.727568,0.602479
3,KNN - SMOTE,0.896365,0.456427,0.901730,0.606077,0.930495,0.531050,0.596354
4,KNN - SMOTE-ENN,0.896365,0.456427,0.901730,0.606077,0.930495,0.531050,0.596354
5,KNN - SMOTE-TOMEK,0.921665,0.535211,0.866352,0.661663,0.945452,0.743144,0.643185


## KNN Experiments Evaluation Results

The table below compares the performance of different KNN experiments on the validation set.

| Experiment | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC |
|---|---:|---:|---:|---:|---:|---:|---:|
| KNN - transformed | `0.9595` | `0.8684` | 0.6384 | `0.7358` | 0.9150 | `0.7652` | `0.7244` |
| KNN - transformed - weights distance | 0.9577 | 0.8360 | 0.6494 | 0.7310 | 0.9145 | 0.7849 | 0.7151 |
| KNN - ADASYN | 0.9024 | 0.4723 | `0.8844` | 0.6158 | 0.9465 | 0.7276 | 0.6023 |
| KNN - SMOTE | 0.8964 | 0.4564 | 0.9017 | 0.6061 | 0.9305 | 0.5311 | 0.5964 |
| KNN - SMOTE-ENN | 0.8964 | 0.4564 | 0.9017 | 0.6061 | 0.9305 | 0.5311 | 0.5964 |
| KNN - SMOTE-TOMEK | 0.9217 | 0.5352 | 0.8664 | 0.6617 | `0.9454` | 0.7431 | 0.6432 |

#### I will continue with ADASYN for hyperparameter tuning.

ADASYN achieves the best **Recall** (0.8844) among the resampled datasets while maintaining the highest **PR-AUC** and **MCC** of the oversampled experiments. SMOTE and SMOTE-ENN produce identical results here and have a notably lower PR-AUC (0.5311), making ADASYN the stronger choice for the positive class.

## Tuning Hyperparameters

| Hyperparameter | Meaning | Importance |
|---|---|---|
| `n_neighbors` | Number of nearest neighbors to consider. | Controls the bias-variance trade-off. Smaller values = more flexible boundary. |
| `weights` | How neighbors contribute to prediction. | `uniform` treats all equally; `distance` gives closer neighbors more influence. |
| `metric` | Distance function used to find neighbors. | Affects how similarity is measured between samples. |
| `p` | Power parameter for Minkowski metric. | `p=1` = Manhattan distance; `p=2` = Euclidean distance. |

#### Experiment 1
Baseline — `n_neighbors=5`, uniform weights, Euclidean distance

In [72]:
knn_tuning_results = []

model1 = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="minkowski",
    p=2,
    n_jobs=-1,
)

model1.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    knn_tuning_results,
    "KNN - E1 Tuning | k=5 | weights=uniform | metric=euclidean",
    model1,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== KNN - E1 Tuning | k=5 | weights=uniform | metric=euclidean ==========
Accuracy: 0.9025
Precision: 0.4725
Recall: 0.8844
F1-score: 0.6159
ROC-AUC: 0.9465
PR-AUC: 0.7276
MCC: 0.6025



#### Experiment 2
Increase `n_neighbors` to reduce noise sensitivity

In [73]:
model2 = KNeighborsClassifier(
    n_neighbors=11,
    weights="uniform",
    metric="minkowski",
    p=2,
    n_jobs=-1,
)

model2.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    knn_tuning_results,
    "KNN - E2 Tuning | k=11 | weights=uniform | metric=euclidean",
    model2,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== KNN - E2 Tuning | k=11 | weights=uniform | metric=euclidean ==========
Accuracy: 0.8714
Precision: 0.4013
Recall: 0.9237
F1-score: 0.5595
ROC-AUC: 0.9610
PR-AUC: 0.7759
MCC: 0.5571



#### Experiment 3
Try `weights=distance` — closer neighbors have more influence

In [74]:
model3 = KNeighborsClassifier(
    n_neighbors=11,
    weights="distance",
    metric="minkowski",
    p=2,
    n_jobs=-1,
)

model3.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    knn_tuning_results,
    "KNN - E3 Tuning | k=11 | weights=distance | metric=euclidean",
    model3,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== KNN - E3 Tuning | k=11 | weights=distance | metric=euclidean ==========
Accuracy: 0.9440
Precision: 0.6255
Recall: 0.9127
F1-score: 0.7423
ROC-AUC: 0.9702
PR-AUC: 0.8229
MCC: 0.7281



#### Experiment 4
Try Manhattan distance (`p=1`)

In [75]:
model4 = KNeighborsClassifier(
    n_neighbors=11,
    weights="distance",
    metric="minkowski",
    p=1,
    n_jobs=-1,
)

model4.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    knn_tuning_results,
    "KNN - E4 Tuning | k=11 | weights=distance | metric=manhattan",
    model4,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== KNN - E4 Tuning | k=11 | weights=distance | metric=manhattan ==========
Accuracy: 0.9463
Precision: 0.6378
Recall: 0.9080
F1-score: 0.7493
ROC-AUC: 0.9713
PR-AUC: 0.8325
MCC: 0.7343



#### Experiment 5
Larger `n_neighbors` with distance weights

In [76]:
model5 = KNeighborsClassifier(
    n_neighbors=21,
    weights="distance",
    metric="minkowski",
    p=2,
    n_jobs=-1,
)

model5.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    knn_tuning_results,
    "KNN - E5 Tuning | k=21 | weights=distance | metric=euclidean",
    model5,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== KNN - E5 Tuning | k=21 | weights=distance | metric=euclidean ==========
Accuracy: 0.9329
Precision: 0.5738
Recall: 0.9355
F1-score: 0.7113
ROC-AUC: 0.9801
PR-AUC: 0.8698
MCC: 0.7017



#### Experiment 6
Try smaller `n_neighbors` with distance weights

In [77]:
model6 = KNeighborsClassifier(
    n_neighbors=7,
    weights="distance",
    metric="minkowski",
    p=2,
    n_jobs=-1,
)

model6.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    knn_tuning_results,
    "KNN - E6 Tuning | k=7 | weights=distance | metric=euclidean",
    model6,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== KNN - E6 Tuning | k=7 | weights=distance | metric=euclidean ==========
Accuracy: 0.9488
Precision: 0.6551
Recall: 0.8899
F1-score: 0.7547
ROC-AUC: 0.9593
PR-AUC: 0.7823
MCC: 0.7375



In [78]:
print(knn_tuning_results)

[{'Experiment': 'KNN - E1 Tuning | k=5 | weights=uniform | metric=euclidean', 'Accuracy': 0.9024814068256064, 'Precision': 0.4724905501889962, 'Recall': 0.8844339622641509, 'F1-score': 0.6159321105940323, 'ROC-AUC': np.float64(0.946535275753674), 'PR-AUC': np.float64(0.7275675622977222), 'MCC': np.float64(0.6024788714163172)}, {'Experiment': 'KNN - E2 Tuning | k=11 | weights=uniform | metric=euclidean', 'Accuracy': 0.8714116911100299, 'Precision': 0.40129781420765026, 'Recall': 0.9237421383647799, 'F1-score': 0.5595238095238095, 'ROC-AUC': np.float64(0.9609617810035559), 'PR-AUC': np.float64(0.7758756417376761), 'MCC': np.float64(0.5571074843979047)}, {'Experiment': 'KNN - E3 Tuning | k=11 | weights=distance | metric=euclidean', 'Accuracy': 0.9439772016403698, 'Precision': 0.6255387931034483, 'Recall': 0.9127358490566038, 'F1-score': 0.7423273657289002, 'ROC-AUC': np.float64(0.9701986778785634), 'PR-AUC': np.float64(0.8228517735374448), 'MCC': np.float64(0.7281394266708091)}, {'Experim

## KNN Hyperparameter Tuning Results

| Experiment | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC |
|---|---:|---:|---:|---:|---:|---:|---:|
| KNN - E1 Tuning \| k=5 \| weights=uniform \| metric=euclidean | 0.9024 | 0.4723 | 0.8844 | 0.6158 | 0.9465 | 0.7276 | 0.6023 |
| KNN - E2 Tuning \| k=11 \| weights=uniform \| metric=euclidean | 0.8714 | 0.4013 | `0.9237` | 0.5595 | 0.9610 | 0.7759 | 0.5571 |
| KNN - E3 Tuning \| k=11 \| weights=distance \| metric=euclidean | 0.9440 | 0.6255 | 0.9127 | 0.7423 | 0.9702 | 0.8229 | 0.7281 |
| KNN - E4 Tuning \| k=11 \| weights=distance \| metric=manhattan | 0.9463 | 0.6378 | 0.9080 | 0.7493 | 0.9713 | 0.8325 | 0.7343 |
| KNN - E5 Tuning \| k=21 \| weights=distance \| metric=euclidean | 0.9329 | 0.5738 | 0.9355 | 0.7113 | `0.9801` | `0.8698` | 0.7017 |
| KNN - E6 Tuning \| k=7 \| weights=distance \| metric=euclidean | `0.9488` | `0.6551` | 0.8899 | `0.7547` | 0.9593 | 0.7823 | `0.7375` |

### Interpretation

Switching from `weights=uniform` to `weights=distance` had the largest positive impact across all metrics. E1 and E2 used uniform weights and produced low precision (0.47 and 0.40), meaning most positive predictions were false alarms despite decent recall.

**E5** (`k=21`, distance, euclidean) achieved the highest **Recall** (0.9355), **ROC-AUC** (0.9801), and **PR-AUC** (0.8698), making it the strongest model for detecting diabetic patients.

**E6** (`k=7`, distance, euclidean) delivered the best **F1-score** (0.7547), **MCC** (0.7375), and **Accuracy** (0.9488), offering the best overall balance.

**E4** (`k=11`, distance, manhattan) came close to E6 but was slightly weaker on F1 and MCC.

Since recall is the top priority in this project (missing a diabetic patient is more harmful than a false alarm), **E5** is selected as the best model.

### Selected Model

Based on the validation results, the best KNN model is:

> **KNN - E5 Tuning | k=21 | weights=distance | metric=euclidean**

This model achieves the highest recall (0.9355), the best ROC-AUC (0.9801), and the highest PR-AUC (0.8698), making it the most reliable model for detecting diabetic patients while maintaining a strong overall score.

### Best Model Results

| Metric | Value |
|---|---:|
| Accuracy | 0.9329 |
| Precision | 0.5738 |
| Recall | 0.9355 |
| F1-score | 0.7113 |
| ROC-AUC | 0.9801 |
| PR-AUC | 0.8698 |
| MCC | 0.7017 |

## Final Evaluation On Test Data

In [79]:
transformer = DataTransformation()
transformer.load_preprocessor()
test = pd.read_csv('../../data/split/test.csv')
x_test = test.drop("diabetes", axis=1)
y_test = test["diabetes"]
x_test_transformed = transformer.transform(x_test)
x_test_transformed = x_test_transformed[x_train_ADASYN.columns]

c:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or

In [80]:
knn_final_result = []

evaluate_and_store(
    knn_final_result,
    "KNN - ADASYN Final",
    model5,  # best tuned model: k=21, weights=distance, metric=euclidean
    x_test_transformed,
    y_test,
)

========== KNN - ADASYN Final ==========
Accuracy: 0.9289
Precision: 0.5590
Recall: 0.9277
F1-score: 0.6976
ROC-AUC: 0.9730
PR-AUC: 0.8591
MCC: 0.6874



In [81]:
joblib.dump(model5, "../../models/KNN_model.pkl")  # best tuned model: k=21, weights=distance, metric=euclidean

['../../models/KNN_model.pkl']